<a href="https://colab.research.google.com/github/lakshay5750/AgenticAI/blob/main/Reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain

In [2]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain_groq]


In [3]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')

In [5]:

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)



# Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    api_key=GROQ_API_KEY
)


def run_reflection_loop():
    """
    Demonstrates a multi-step AI reflection loop
    to progressively improve a Python function.
    """

    # --- Core Task ---

    task_prompt = """
    Your task is to create a Python function named
    calculate_factorial.

    This function should do the following:

    1. Accept a single integer n as input.
    2. Calculate its factorial (n!).
    3. Include a clear docstring explaining what
       the function does.
    4. Handle edge cases: The factorial of 0 is 1.
    5. Handle invalid input: Raise a ValueError if
       the input is a negative number.
    """

    # --- Reflection Loop ---

    max_iterations = 3
    current_code = ""

    # Conversation history
    message_history = [
        HumanMessage(content=task_prompt)
    ]

    for i in range(max_iterations):

        print(
            "\n" + "=" * 25 +
            f" REFLECTION LOOP: ITERATION {i + 1} " +
            "=" * 25
        )

        # --- 1. GENERATE / REFINE STAGE ---

        if i == 0:

            print("\n>>> STAGE 1: GENERATING initial code...")

            response = llm.invoke(message_history)
            current_code = response.content

        else:

            print("\n>>> STAGE 1: REFINING code based on previous critique...")

            message_history.append(
                HumanMessage(
                    content="Please refine the code using the critiques provided."
                )
            )

            response = llm.invoke(message_history)
            current_code = response.content

        print(
            "\n--- Generated Code (v" +
            str(i + 1) + ") ---\n" +
            current_code
        )

        # Add generated code to history
        message_history.append(response)

        # --- 2. REFLECT STAGE ---

        print("\n>>> STAGE 2: REFLECTING on the generated code...")

        reflector_prompt = [

            SystemMessage(content="""
                You are a senior software engineer
                and an expert in Python.

                Your role is to perform a meticulous
                code review.

                Critically evaluate the provided Python
                code based on the original task requirements.

                Look for bugs, style issues, missing edge cases,
                and areas for improvement.

                If the code is perfect and meets all requirements,
                respond with the single phrase 'CODE_IS_PERFECT'.

                Otherwise, provide a bulleted list of your critiques.
            """),

            HumanMessage(
                content=f"""
                Original Task:
                {task_prompt}

                Code to Review:
                {current_code}
                """
            )
        ]

        critique_response = llm.invoke(reflector_prompt)

        critique = critique_response.content

        # --- 3. STOPPING CONDITION ---

        if "CODE_IS_PERFECT" in critique:

            print(
                "\n--- Critique ---\n"
                "No further critiques found. "
                "The code is satisfactory."
            )

            break

        print("\n--- Critique ---\n" + critique)

        # Add critique to conversation history
        message_history.append(
            HumanMessage(
                content=f"""
                Critique of the previous code:
                {critique}
                """
            )
        )

    # --- FINAL RESULT ---

    print("\n" + "=" * 30 + " FINAL RESULT " + "=" * 30)

    print(
        "\nFinal refined code after the reflection process:\n"
    )

    print(current_code)


if __name__ == "__main__":
    run_reflection_loop()



========================= REFLECTION LOOP: ITERATION 1 =========================

>>> STAGE 1: GENERATING initial code...

--- Generated Code (v1) ---
Here is a clean, well‑documented implementation that follows all the requirements.

```python
def calculate_factorial(n: int) -> int:
    """
    Calculate the factorial of a non‑negative integer.

    The factorial of n (written n!) is the product of all positive integers
    from 1 up to n. By definition, 0! = 1.

    Parameters
    ----------
    n : int
        The integer whose factorial is to be computed. Must be >= 0.

    Returns
    -------
    int
        The factorial of ``n``.

    Raises
    ------
    ValueError
        If ``n`` is negative.
    TypeError
        If ``n`` is not an integer.

    Examples
    --------
    >>> calculate_factorial(5)
    120
    >>> calculate_factorial(0)
    1
    >>> calculate_factorial(-3)
    Traceback (most recent call last):
        ...
    ValueError: factorial is not defined for negat

In [7]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')

In [18]:

import asyncio
import os

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# --------------------------------
# Configuration
# --------------------------------

load_dotenv()

# ADK expects GOOGLE_API_KEY for Gemini API authentication.
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("Gemini API key not found in .env file")

os.environ["GOOGLE_API_KEY"] = api_key

MODEL = "gemini-3.8-flash"


# --------------------------------
# Agent 1: Draft Generator
# --------------------------------

generator = LlmAgent(
    name="DraftWriter",
    model=MODEL,
    description="Generates initial draft content.",
    instruction="""
    Write a short, informative paragraph about the user's subject.
    Make it clear, informative and easy to understand.
    """,
    output_key="draft_text"
)


# --------------------------------
# Agent 2: Fact Checker
# --------------------------------

reviewer = LlmAgent(
    name="FactChecker",
    model=MODEL,
    description="Reviews a draft for factual accuracy.",
    instruction="""
    You are a meticulous fact-checker.

    Read the draft stored in the state key 'draft_text'.

    Carefully evaluate its factual accuracy.

    Return a JSON object containing exactly two keys:

    {
        "status": "ACCURATE or INACCURATE",
        "reasoning": "Explanation of your decision"
    }

    Return only valid JSON without Markdown formatting.

    Draft to review:

    {draft_text}
    """,
    output_key="review_output"
)


# --------------------------------
# Helper function to print events
# --------------------------------

def print_event(event):

    if event.is_final_response():

        if event.content and event.content.parts:

            for part in event.content.parts:

                if part.text:
                    print(part.text)

        else:
            print("Final event received without text content.")


# --------------------------------
# Main execution
# --------------------------------

async def main():

    session_service = InMemorySessionService()

    session = await session_service.create_session(
        app_name="ReviewApp",
        user_id="user1"
    )

    # ==================================
    # STEP 1: Run Draft Generator
    # ==================================

    print("\n========== DRAFT GENERATOR ==========\n")

    generator_runner = Runner(
        agent=generator,
        app_name="ReviewApp",
        session_service=session_service
    )

    user_message = types.Content(
        role="user",
        parts=[
            types.Part(text="Artificial Intelligence")
        ]
    )

    try:

        async for event in generator_runner.run_async(
            user_id="user1",
            session_id=session.id,
            new_message=user_message
        ):

            print_event(event)

    except Exception as e:

        print("Generator Error:", e)
        return

    # ==================================
    # STEP 2: Run Fact Checker
    # ==================================

    print("\n========== FACT CHECKER ==========\n")

    reviewer_runner = Runner(
        agent=reviewer,
        app_name="ReviewApp",
        session_service=session_service
    )

    review_message = types.Content(
        role="user",
        parts=[
            types.Part(
                text="Please review the generated draft."
            )
        ]
    )

    try:

        async for event in reviewer_runner.run_async(
            user_id="user1",
            session_id=session.id,
            new_message=review_message
        ):

            print_event(event)

    except Exception as e:

        print("Reviewer Error:", e)
        return

    # ==================================
    # STEP 3: Display Shared State
    # ==================================

    print("\n========== SESSION STATE ==========\n")

    updated_session = await session_service.get_session(
        app_name="ReviewApp",
        user_id="user1",
        session_id=session.id
    )

    print("Draft Text:")
    print(updated_session.state.get("draft_text"))

    print("\nReview Output:")
    print(updated_session.state.get("review_output"))


# Google Colab execution
await main()



========== DRAFT GENERATOR ==========



CancelledError: 